## 一些数学技巧
依托 pytorch 工具库实现的数学定理的应用示范。

In [100]:
import torch

### 矩阵/张量 | `.tensor()`
- 矩阵，是数据容器，是张量的特殊形式。
- 张量，是矩阵的高纬扩展

``` txt
张量是矩阵的高维扩展，可以理解为：
0 维张量 = 标量（单个数字，比如 5、3.14）；
1 维张量 = 向量（一维数组，比如 [1,2,3]）；
2 维张量 = 矩阵（二维表格，就是上面说的矩阵）；
3 维张量 = 多个矩阵堆叠（比如 4 个 3×2 矩阵 → 形状 (4,3,2)）；
4 维张量 = 多个 3 维张量堆叠（比如 2 个 4×3×2 张量 → 形状 (2,4,3,2)）；
用 “装箱” 比喻更易理解：
标量（0 维）：单个苹果；
向量（1 维）：一排苹果（1 列）；
矩阵（2 维）：苹果摆成的表格（多行多列）；
3 维张量：多个苹果表格叠成的立方体；
4 维张量：多个立方体装成一箱；
```


In [95]:
matrix = torch.tensor([[1,2], [4,5], [7,8]], dtype=torch.float)
print("矩阵维度：", matrix.ndim)  # 2
print("矩阵形状：", matrix.shape) # torch.Size([3,2]) → (行数, 列数)
print("取第2行第1列元素：", matrix[1,0])  # tensor(4.) → 索引从0开始

矩阵维度： 2
矩阵形状： torch.Size([3, 2])
取第2行第1列元素： tensor(4.)


In [98]:
a = torch.tensor([
    [1,2,3],
    [4,5,6],
    [7,8,9]
], dtype=torch.float)
print(a.shape)
print(a)

torch.Size([3, 3])
tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]])


### 求平均 | `.mean()`

In [99]:
print(torch.mean(a)) # 所有元素求平均
print(torch.mean(a, dim=0)) # 沿着 0 维（行）求平均，列维度保留。
print(torch.mean(a, dim=1)) # 沿着 1 维（列）求平均，行维度保留。
print(torch.mean(a, dim=0, keepdim=True))
print(torch.mean(a, dim=1, keepdim=True))

tensor(5.)
tensor([4., 5., 6.])
tensor([2., 5., 8.])
tensor([[4., 5., 6.]])
tensor([[2.],
        [5.],
        [8.]])


### 相似度校验 | `.allclose()`

In [94]:
'''
torch.allclose(a, b) 的判断公式（简化版）：
∣a−b∣≤atol+rtol×∣b∣

* 默认参数：rtol=1e-05（相对误差阈值）、atol=1e-08（绝对误差阈值）；可手工设置，如 torch.allclose(a, b, rtol=1e-6, atol=1e-7)
* 只有当所有元素都满足上述不等式时，才返回 True，否则返回 False；
* 关键：对大数，误差判断以「相对误差」为主；对小数，以「绝对误差」为主。

* 我们逐元素验证不等式：
1. 第一个元素：10000.0 vs 10000.1
差值：|10000 - 10000.1| = 0.1
计算阈值：atol + rtol × |b| = 1e-08 + 1e-05 × 10000.1 = 1e-08 + 0.100001 ≈ 0.100001
比较：0.1 ≤ 0.100001 → ✅ 满足
2. 第二个元素：1e-07 (0.0000001) vs 1e-08 (0.00000001)
差值：|1e-07 - 1e-08| = 9e-08
计算阈值：atol + rtol × |b| = 1e-08 + 1e-05 × 1e-08 = 1e-08 + 1e-13 ≈ 1e-08
比较：9e-08 ≤ 1e-08 → ❌ 不满足

* 结论：只要有一个元素不满足，整体返回 False。
'''

a = torch.tensor([10000., 1e-07])
b = torch.tensor([10000.1, 1e-08])
torch.allclose(a, b)


False

### 矩阵乘积 | `a @ b`

In [57]:
torch.manual_seed(42)
a = torch.tensor([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
], dtype=torch.float)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b # 只有 a 的列与 b 的行一致，就可以进行矩阵乘法，结果的shape就是 a 的行 x b 的列。
print('a=')
print(a)
print('----------')
print('b=')
print(b)
print('----------')
print('a @ b=')
'''
1. 计算 c [0][0]（第一行第一列）
a 第一行：[1, 2, 3]
b 第一列：[2, 6, 6]
计算：1×2 + 2×6 + 3×6 = 2 + 12 + 18 = 32 → 对应结果 32.
2. 计算 c [0][1]（第一行第二列）
a 第一行：[1, 2, 3]
b 第二列：[7, 4, 5]
计算：1×7 + 2×4 + 3×5 = 7 + 8 + 15 = 30 → 对应结果 30.
3. 计算 c [1][0]（第二行第一列）
a 第二行：[4, 5, 6]
b 第一列：[2, 6, 6]
计算：4×2 + 5×6 + 6×6 = 8 + 30 + 36 = 74 → 对应结果 74.
4. 计算 c [1][1]（第二行第二列）
a 第二行：[4, 5, 6]
b 第二列：[7, 4, 5]
计算：4×7 + 5×4 + 6×5 = 28 + 20 + 30 = 78 → 对应结果 78.
5. 计算 c [2][0]（第三行第一列）
a 第三行：[7, 8, 9]
b 第一列：[2, 6, 6]
计算：7×2 + 8×6 + 9×6 = 14 + 48 + 54 = 116 → 对应结果 116.
6. 计算 c [2][1]（第三行第二列）
a 第三行：[7, 8, 9]
b 第二列：[7, 4, 5]
计算：7×7 + 8×4 + 9×5 = 49 + 32 + 45 = 126 → 对应结果 126.
'''
print(c)

a=
tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]])
----------
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
----------
a @ b=
tensor([[ 32.,  30.],
        [ 74.,  78.],
        [116., 126.]])


In [92]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b
print('a=')
print(a)
print('----------')
print('b=')
print(b)
print('----------')
print('a @ b=')
print(c)

a=
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
----------
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
----------
a @ b=
tensor([[ 2.,  7.],
        [ 8., 11.],
        [14., 16.]])


In [93]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, 1, True)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b
print('a=')
print(a)
print('----------')
print('b=')
print(b)
print('----------')
print('a @ b=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
----------
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
----------
a @ b=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


### 矩阵转置 | `transporse`
PyTorch 中张量转置有两种常用方式：`tensor.T`（简化版，仅适用于 2 维张量）和 `torch.transpose()`（通用版，支持任意维度）

In [71]:
a = torch.tensor([
    [1, 2],
    [3, 4],
    [5, 6]
], dtype=torch.float)
a = a.T
print('a=')
print(a)


a=
tensor([[1., 3., 5.],
        [2., 4., 6.]])


In [90]:
torch.manual_seed(9527)

K = torch.randn(2, 4, 6)

k_t = K.transpose(-2, -1)

print(k_t.shape)
print(torch.allclose(K.transpose(-2, -1), K.transpose(-1, -2)))
print(torch.allclose(K.transpose(1, 2), K.transpose(2, 1)))
print(torch.allclose(K.transpose(-2, -1), K.transpose(1, 2)))

torch.Size([2, 6, 4])
True
True
True


In [91]:
torch.manual_seed(9527)

Q = torch.randn(2, 4, 6)
K = torch.randn(2, 4, 6)

wei = Q @ K.transpose(-2, -1)

wei.shape

torch.Size([2, 4, 4])

### 三角矩阵 | `.tril()`

In [9]:
torch.tril(torch.ones(3,3))
# 生成下三角矩阵【torch.triu 则生成上三角矩阵】
# tensor([[1., 0., 0.],
#         [1., 1., 0.],
#         [1., 1., 1.]])

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

### 点积/内积 | `.sum()`

In [11]:
a = torch.tensor([
    [1, 2],
    [4, 5],
    [7, 8]
], dtype=torch.float)
torch.sum(a, 0) # tensor([12., 15.])
torch.sum(a, 0, True) # tensor([[12., 15.]]) # 应用于 batchNorm 之中

torch.sum(a, 1) # tensor([ 3.,  9., 15.])
torch.sum(a, 1, True) # tensor([[ 3.], # 应用于 layerNorm 之中
                       #        [ 9.],
                       #        [15.]])

a / torch.sum(a, 0, True) # ❌ tensor([[0.0833, 0.1333],
                        #           [0.3333, 0.3333],
                        #           [0.5833, 0.5333]])

a / torch.sum(a, 1, True) # ✅ tensor([[0.3333, 0.6667], # 应用于 softmax 之中
                        #           [0.4444, 0.5556],
                        #           [0.4667, 0.5333]])

tensor([[0.3333, 0.6667],
        [0.4444, 0.5556],
        [0.4667, 0.5333]])

### 合并同结构张量 | `.cat()`

`torch.cat(tensors, dim=0, out=None)`
- tensors: 待拼接的张量列表
- dim: 指定拼接的维度
- out: 可选，指定输出张量(一般不用，默认返回新张量)


``` txt
维度 0 拼接（纵向）：把多列 “学生成绩单”（每行是一个学生）按行堆叠，变成更长的成绩单；
维度 1 拼接（横向）：给同一份成绩单增加 “总分” 列，变成更宽的成绩单；
高维拼接（如 Transformer 的多头注意力）：把多个注意力头的输出沿特征维度拼接，恢复完整特征。

工程技巧
原地拼接（节省内存）：若需节省内存，可指定 out 参数（但极少用，PyTorch 自动优化）
out = torch.empty(4,4)  # 预分配输出空间
torch.cat([A,B], dim=1, out=out)
```

In [106]:
# shape: [2, 2]
a = torch.tensor([
    [1, 2],
    [3, 4]
])

# shape: [2, 2]
b = torch.tensor([
    [5, 6],
    [7, 8]
])

# shape: [4, 2]
out = torch.cat([a, b], dim=0)
out


tensor([[1, 2],
        [3, 4],
        [5, 6],
        [7, 8]])

In [110]:
# shape: [2, 2]
a = torch.tensor([
    [1, 2],
    [3, 4]
])

# shape: [2, 2]
b = torch.tensor([
    [5, 6],
    [7, 8]
])

# shape: [4, 2]
out = torch.cat([a, b], dim=-1)
out


tensor([[1, 2, 5, 6],
        [3, 4, 7, 8]])

### 合并多个样本 | `.stack()`
区分于 `.cat()` 如下：
``` txt
torch.cat      拼接，不新增维度       dim=0 → (4,2)，dim=1 → (2,4)	       合并同结构的张量（如多头输出）
torch.stack    堆叠，新增一个维度      dim=0 → (2,2,2)，dim=1 → (2,2,2)	   新增维度分组（如合并多个样本）
```

In [120]:
# shape: [2, 2]
a = torch.tensor([
    [1, 2],
    [3, 4]
])

# shape: [2, 2]
b = torch.tensor([
    [5, 6],
    [7, 8]
])

# shape: [2, 2, 2]
out = torch.stack([a, b], dim=0)
out

tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])

In [119]:
# shape: [2, 2]
a = torch.tensor([
    [1, 2],
    [3, 4]
])

# shape: [2, 2]
b = torch.tensor([
    [5, 6],
    [7, 8]
])

# shape: [2, 2, 2]
out = torch.stack([a, b], dim=-1)
out

tensor([[[1, 5],
         [2, 6]],

        [[3, 7],
         [4, 8]]])

## 线性层（全连接层） | `nn.Linear()`
nn.Linear 是 transformer 注意力机制中生成 Q/K/V 向量的核心组件

一、先明确核心功能
`nn.Linear(in_features, out_features, bias=True)` 的本质是：对输入张量做「矩阵乘法 + 偏置（可选）」的线性变换。公式为：
$$
y = x \cdot W^T + b
$$
当 `bias=False` 时，公式简化为：
$y = x \cdot W^T + b$
 
这是深度学习中 “特征投影 / 维度变换” 的基础操作，无激活函数时输出和输入呈线性关系。

In [36]:
import torch.nn as nn

In [39]:
linear = nn.Linear(14, 40, bias=False)
x = torch.randn(12, 13, 14)
out = linear(x)
out.shape

torch.Size([12, 13, 40])

In [26]:
B, T, C = 4, 8, 32
head_size = 16

# 1. 定义线性层：将 32 维输入投影为 16 维输出，无偏置
linear_q = nn.Linear(C, head_size, bias=False)

# 2. 模拟输入：batch=4个样本，T=8个时间步，C=32维度
x = torch.randn(4, 8, 32)  # shape: (4,8,32)

# 3. 执行线性变换（生成Q向量）
Q = linear_q(x)  # shape: (4,8,16)

print(Q.shape)

torch.Size([4, 8, 16])
